# Vector Stores in LangChain

Roadmap of what to learn:

| # | Concept | Why it matters |
|---|---|---|
| 1 | `add_documents`, `get`, `get_by_ids`, `update_documents` | basic CRUD |
| 2 | `similarity_search`, `similarity_search_with_relevance_scores` | basic search |
| 3 | Embeddings, dimensions, cosine similarity | what "similar" actually means |
| 4 | Custom IDs | no duplicates when you re-run, easy update / delete |
| 5 | Scores: distance vs relevance | lower or higher = better? |
| 6 | Metadata filtering | search only inside some documents (per user, per file, per year) |
| 7 | Full-text filter (`where_document`) | keyword must appear in the chunk |
| 8 | MMR search | relevant **and** diverse results |
| 9 | Search by vector | reuse an embedding you already have |
| 10 | `delete` | remove by id or by filter |
| 11 | `as_retriever()` | how vector stores plug into RAG chains / LangGraph |
| 12 | Persistence | load an existing store instead of re-embedding |
| 13 | Async methods | for web apps / APIs |
| 14 | Hybrid search (BM25 + vectors) | exact keywords + meaning, common in production |
| 15 | Same interface, different stores | swap Chroma for InMemory / FAISS / Pinecone / MongoDB Atlas |
| 16 | Common pitfalls | bugs that fail silently |

In [1]:
from dotenv import load_dotenv
load_dotenv()   # OPENAI_API_KEY from .env

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings()   # text-embedding-ada-002 by default

docs = [
    Document(page_content="Apples are red or green fruits rich in fiber.",        metadata={"category": "fruit",     "year": 2021, "source": "fruits.txt"}),
    Document(page_content="Bananas are yellow fruits full of potassium.",         metadata={"category": "fruit",     "year": 2022, "source": "fruits.txt"}),
    Document(page_content="Mangoes are sweet tropical fruits, the king of fruits.", metadata={"category": "fruit",   "year": 2023, "source": "fruits.txt"}),
    Document(page_content="Mango is a tropical fruit known for its sweetness.",   metadata={"category": "fruit",     "year": 2024, "source": "fruits.txt"}),   # near duplicate, for MMR
    Document(page_content="Carrots are orange vegetables good for the eyes.",     metadata={"category": "vegetable", "year": 2022, "source": "veggies.txt"}),
    Document(page_content="Spinach is a green leafy vegetable rich in iron.",     metadata={"category": "vegetable", "year": 2023, "source": "veggies.txt"}),
    Document(page_content="Python is a popular programming language for AI.",     metadata={"category": "tech",      "year": 2024, "source": "tech.txt"}),
    Document(page_content="Apple Inc. makes the iPhone and MacBook.",             metadata={"category": "tech",      "year": 2023, "source": "tech.txt"}),
]
ids = [f"doc_{i}" for i in range(len(docs))]   # our own ids

store = Chroma(
    collection_name="vs_concepts",
    embedding_function=embeddings,
    persist_directory="chroma_db",
)

/Users/sriharshamadireddy/Desktop/Personal/Learning/DataScience/genai/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Basic CRUD
- `add_documents(docs, ids=...)` → embeds and stores, returns the ids
- `get(ids)` → raw dictionary (ids, documents, metadatas)
- `get_by_ids([...])` → list of `Document`
- `update_document(id, doc)` / `update_documents([ids], [docs])` → replace text + metadata (re-embeds)

(Why we pass our own `ids` is explained in section 4.)

In [2]:
store.add_documents(docs, ids=ids)

print(store.get("doc_0", include=['documents']))            # dictionary
print(store.get_by_ids(["doc_0"]))   # list of Documents

{'ids': ['doc_0'], 'embeddings': None, 'documents': ['Apples are red or green fruits rich in fiber.'], 'uris': None, 'included': ['documents'], 'data': None, 'metadatas': None}
[Document(id='doc_0', metadata={'category': 'fruit', 'year': 2021, 'source': 'fruits.txt'}, page_content='Apples are red or green fruits rich in fiber.')]


In [ ]:
# update one
store.update_document("doc_0", Document(page_content="Apples keep the doctor away.", metadata={"category": "fruit", "year": 2025, "source": "fruits.txt"}))
print(store.get_by_ids(["doc_0"]))

# update many: list of ids, list of Documents (here: put the originals back)
store.update_documents(["doc_0"], [docs[0]])
print(store.get_by_ids(["doc_0"]))

## 2. Basic search
- `similarity_search(query, k)` → the `k` most similar Documents
- `similarity_search_with_relevance_scores(query, k)` → (Document, score) pairs, score 0-1, higher = more similar

In [5]:
print(store.similarity_search("mango", k=2))

for doc, score in store.similarity_search_with_relevance_scores("what is apple", k=3):
    print(f"{score:.3f}  {doc.page_content}")

[Document(id='doc_3', metadata={'category': 'fruit', 'source': 'fruits.txt', 'year': 2024}, page_content='Mango is a tropical fruit known for its sweetness.'), Document(id='doc_2', metadata={'year': 2023, 'category': 'fruit', 'source': 'fruits.txt'}, page_content='Mangoes are sweet tropical fruits, the king of fruits.')]
0.751  Apple Inc. makes the iPhone and MacBook.
0.727  Apples are red or green fruits rich in fiber.
0.642  Bananas are yellow fruits full of potassium.


## 3. Embeddings, dimensions, cosine similarity
An embedding is a list of numbers that captures meaning. Every vector from the same model has the **same length (dimensions)**.
Similar meaning → vectors point in a similar direction → **cosine similarity** close to 1.

In [ ]:
import numpy as np

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

v_mango  = embeddings.embed_query("mango")
v_banana = embeddings.embed_query("banana")
v_python = embeddings.embed_query("python programming")

print("dimensions:", len(v_mango))
print("mango vs banana :", round(cosine(v_mango, v_banana), 3))   # higher: both fruits
print("mango vs python :", round(cosine(v_mango, v_python), 3))   # lower: unrelated

## 4. Custom IDs
Without ids, every `add_documents` call creates **new** ids → re-running the cell duplicates your data.
With your own ids, adding the same id again **overwrites** it (an upsert). Run the next cell twice: the count stays the same.

In [ ]:
store.add_documents(docs, ids=ids)
print("count:", len(store.get()["ids"]))

## 5. Scores: distance vs relevance
- `similarity_search_with_score` → **distance**, **lower = more similar** (Chroma default)
- `similarity_search_with_relevance_scores` → normalised **0 to 1**, **higher = more similar**

Other stores (Pinecone, MongoDB Atlas) return similarity where higher is better - always check before filtering by score.

In [6]:
print("with_score (distance, lower is better):")
for doc, score in store.similarity_search_with_score("sweet tropical fruit", k=3):
    print(f"  {score:.3f}  {doc.page_content}")

print("\nwith_relevance_scores (0-1, higher is better):")
for doc, score in store.similarity_search_with_relevance_scores("sweet tropical fruit", k=3):
    print(f"  {score:.3f}  {doc.page_content}")

with_score (distance, lower is better):
  0.214  Mango is a tropical fruit known for its sweetness.
  0.218  Mangoes are sweet tropical fruits, the king of fruits.
  0.375  Bananas are yellow fruits full of potassium.

with_relevance_scores (0-1, higher is better):
  0.849  Mango is a tropical fruit known for its sweetness.
  0.846  Mangoes are sweet tropical fruits, the king of fruits.
  0.735  Bananas are yellow fruits full of potassium.


## 6. Metadata filtering
`filter=` searches only the chunks whose metadata matches.
Chroma operators: `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`, and `$and` / `$or` to combine.

In [ ]:
# only vegetables
print(store.similarity_search("healthy food", k=3, filter={"category": "vegetable"}))

# fruits from 2023 onwards ($and is needed for more than one condition)
print(store.similarity_search(
    "fruit",
    k=3,
    filter={"$and": [{"category": "fruit"}, {"year": {"$gte": 2023}}]},
))

# category is fruit OR tech
print(store.similarity_search("apple", k=4, filter={"category": {"$in": ["fruit", "tech"]}}))

## 7. Full-text filter (`where_document`)
Filters on the **text** of the chunk, not the metadata. Useful when an exact word must be present.

In [7]:
# meaning search, but only chunks that contain the word "iPhone"
store.similarity_search("apple", k=3, where_document={"$contains": "iPhone"})

[Document(id='doc_7', metadata={'year': 2023, 'source': 'tech.txt', 'category': 'tech'}, page_content='Apple Inc. makes the iPhone and MacBook.')]

## 8. MMR - Max Marginal Relevance
Plain similarity search can return near duplicates (our two mango chunks).
MMR first fetches `fetch_k` candidates, then picks `k` that are relevant **and** different from each other.
`lambda_mult`: 1 = only relevance, 0 = only diversity.

In [10]:
print("similarity:")
for d in store.similarity_search("mango", k=3):
    print("  ", d.page_content)

print("\nMMR:")
for d in store.max_marginal_relevance_search("mango", k=3, fetch_k=8, lambda_mult=0.4):
    print("  ", d.page_content)

similarity:
   Mango is a tropical fruit known for its sweetness.
   Mangoes are sweet tropical fruits, the king of fruits.
   Bananas are yellow fruits full of potassium.

MMR:
   Mango is a tropical fruit known for its sweetness.
   Apple Inc. makes the iPhone and MacBook.
   Python is a popular programming language for AI.


## 9. Search by vector
If you already have an embedding (cached, or computed elsewhere), search with it directly - no extra embedding call.

In [16]:
query_vector = embeddings.embed_query("leafy greens")
store.similarity_search_by_vector(query_vector, k=2)

[Document(id='doc_5', metadata={'year': 2023, 'category': 'vegetable', 'source': 'veggies.txt'}, page_content='Spinach is a green leafy vegetable rich in iron.'),
 Document(id='doc_0', metadata={'source': 'fruits.txt', 'year': 2021, 'category': 'fruit'}, page_content='Apples are red or green fruits rich in fiber.')]

## 10. get() with filters, and delete
`get()` reads without searching (no embedding call). `include` picks what to return; `where` filters by metadata.

In [15]:
# read only tech docs, with their metadata
print(store.get(where={"category": "tech"}, include=["documents", "metadatas"]))

# delete by id
store.delete(ids=["doc_7"])
print("after delete by id:", len(store.get()["ids"]))

# put it back (same id, so no duplicate)
store.add_documents([docs[7]], ids=["doc_7"])
print("after re-add:", len(store.get()["ids"]))

{'ids': ['doc_6', 'doc_7'], 'embeddings': None, 'documents': ['Python is a popular programming language for AI.', 'Apple Inc. makes the iPhone and MacBook.'], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [{'category': 'tech', 'source': 'tech.txt', 'year': 2024}, {'source': 'tech.txt', 'year': 2023, 'category': 'tech'}]}
after delete by id: 7
after re-add: 8


## 11. `as_retriever()` - the bridge to RAG
A **retriever** has one job: question in → documents out (`.invoke()`).
Chains, agents and LangGraph nodes take retrievers, not vector stores.

| `search_type` | `search_kwargs` |
|---|---|
| `"similarity"` (default) | `k`, `filter` |
| `"mmr"` | `k`, `fetch_k`, `lambda_mult`, `filter` |
| `"similarity_score_threshold"` | `score_threshold` (0-1), `k` |

In [ ]:
retriever = store.as_retriever(search_type="similarity", search_kwargs={"k": 2})
print(retriever.invoke("vegetables for good eyesight"))

mmr_retriever = store.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 8})
print(mmr_retriever.invoke("mango"))

threshold_retriever = store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.8},   # only return good matches, may return nothing
)
print(threshold_retriever.invoke("smartphones"))

filtered_retriever = store.as_retriever(search_kwargs={"k": 2, "filter": {"source": "fruits.txt"}})
print(filtered_retriever.invoke("yellow"))

# many questions at once
print(retriever.batch(["iron rich food", "AI language"]))

## 12. Persistence
With `persist_directory`, data is saved to disk. To use it later, open the store with the **same** `collection_name`, `persist_directory` **and embedding model** - nothing is re-embedded.

In [ ]:
reloaded = Chroma(
    collection_name="vs_concepts",
    embedding_function=embeddings,
    persist_directory="chroma_db",
)
print("docs on disk:", len(reloaded.get()["ids"]))
reloaded.similarity_search("fiber", k=1)

## 13. Async methods
Most methods have an `a` version (`aadd_documents`, `asimilarity_search`, retriever `ainvoke`).
Use them in async apps (FastAPI, async LangGraph) so one request doesn't block others.

In [ ]:
results = await store.asimilarity_search("potassium", k=1)
print(results)

print(await retriever.ainvoke("orange vegetable"))

## 14. Hybrid search (keywords + meaning)
- **Vector search** understands meaning ("car" finds "automobile") but can miss exact terms (product codes, names).
- **BM25** is classic keyword search: great for exact words, no understanding of meaning.

`EnsembleRetriever` runs both and merges the rankings. Very common in production RAG. Needs `pip install rank_bm25`.

In [17]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

bm25_retriever = BM25Retriever.from_documents(docs, k=2)        # keyword search, in memory
vector_retriever = store.as_retriever(search_kwargs={"k": 2})   # meaning search

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)
hybrid_retriever.invoke("iPhone")

/var/folders/hl/d4cjwg3x7bs25zps_ddj_tzw0000gn/T/ipykernel_44605/3464890267.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


[Document(metadata={'category': 'tech', 'year': 2023, 'source': 'tech.txt'}, page_content='Apple Inc. makes the iPhone and MacBook.'),
 Document(metadata={'category': 'tech', 'year': 2024, 'source': 'tech.txt'}, page_content='Python is a popular programming language for AI.'),
 Document(id='doc_0', metadata={'year': 2021, 'category': 'fruit', 'source': 'fruits.txt'}, page_content='Apples are red or green fruits rich in fiber.')]

## 15. Same interface, different stores
Every LangChain vector store has the same methods, so switching is mostly a one-line change.
`InMemoryVectorStore` needs no install - handy for tests and quick experiments (data is lost when the kernel stops).

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

memory_store = InMemoryVectorStore(embedding=embeddings)
memory_store.add_documents(docs, ids=ids)
memory_store.similarity_search("tropical fruit", k=2)

| Store | Type | Good for |
|---|---|---|
| `InMemoryVectorStore` | in memory | tests, demos |
| Chroma | local / server | learning, small - medium apps |
| FAISS | local library | fast search on one machine (`pip install faiss-cpu`) |
| pgvector | Postgres extension | you already use Postgres |
| MongoDB Atlas | managed | you already use MongoDB |
| Pinecone, Qdrant, Weaviate, Milvus | managed / self hosted | large scale production |

## 16. Common pitfalls
1. **Different embedding model for indexing and querying** → search returns garbage, no error. Always reopen a store with the same model.
2. **No custom ids** → duplicates every time you re-run the notebook.
3. **Score direction** → distance (lower better) vs similarity (higher better) differs per store and per method.
4. **Chunks too small or too big** → tiny chunks lose context, huge chunks dilute meaning. 300-1000 characters is a common start.
5. **Dimension mismatch** → stores like MongoDB Atlas / Pinecone need the index dimension to match the model (1536 for OpenAI ada-002 / 3-small, 384 for MiniLM).
6. **Metadata not stored** → you can't filter later, so add `source`, `page`, `user_id`, dates when you load documents.

### Next to explore
See the **Advanced** section below: Indexing API, re-ranking, multi-query and parent document retrievers.

# Advanced

| # | Topic | Problem it solves |
|---|---|---|
| A1 | Indexing API (`index()` + `SQLRecordManager`) | re-running ingestion re-embeds everything and leaves stale chunks behind |
| A2 | Re-ranking (cross-encoder) | vector search is fast but its top results are not always the best ones |
| A3 | Multi-query retriever | one phrasing of a question misses chunks written in other words |
| A4 | Parent document retriever | small chunks search well but give the LLM too little context |

## A1. Indexing API - keep the store in sync with your sources
In real apps your files change. Re-adding everything costs embedding calls and leaves old versions of chunks in the store.

`index()` uses a **record manager** (a small SQL table) that remembers a hash of every chunk it wrote:
- unchanged chunk → **skipped** (no embedding call)
- changed chunk → old one **deleted**, new one **added**
- chunk whose source was removed → **deleted** (depends on `cleanup`)

| `cleanup` | Deletes |
|---|---|
| `None` | nothing, only adds / skips |
| `"incremental"` | old chunks of the **sources you passed in this run** that weren't re-written |
| `"full"` | **everything** not passed in this run - always pass your full dataset |

Every document needs a `source_id_key` in its metadata (we use `source`).

In [ ]:
from langchain_classic.indexes import SQLRecordManager
from langchain_core.indexing import index

index_store = Chroma(collection_name="vs_index_demo", embedding_function=embeddings)   # in memory
index_store.reset_collection()   # start clean every time this demo runs

record_manager = SQLRecordManager(
    namespace="chroma/vs_index_demo",   # one namespace per vector store collection
    db_url="sqlite:///:memory:",        # demo only; in real apps use a file, e.g. "sqlite:///record_manager.sql",
)                                       # so it remembers what was indexed across runs
record_manager.create_schema()

# 1st run: everything is new
print(index(docs, record_manager, index_store, cleanup="incremental", source_id_key="source"))

In [ ]:
# 2nd run, same docs: nothing is embedded again
print(index(docs, record_manager, index_store, cleanup="incremental", source_id_key="source"))

In [ ]:
# change one banana chunk in fruits.txt and re-index the whole dataset
changed_docs = list(docs)
changed_docs[1] = Document(page_content="Bananas are yellow fruits, great before a workout.", metadata=docs[1].metadata)

print(index(changed_docs, record_manager, index_store, cleanup="incremental", source_id_key="source"))
# -> 1 added (new banana), 1 deleted (old banana), 7 skipped

In [ ]:
# "full" cleanup: veggies.txt and tech.txt are not in this run, so their chunks are removed
fruit_docs = [d for d in changed_docs if d.metadata["source"] == "fruits.txt"]
print(index(fruit_docs, record_manager, index_store, cleanup="full", source_id_key="source"))
print("left in store:", len(index_store.get()["ids"]))

## A2. Re-ranking
Vector search compares two **separately** made embeddings - fast, but rough.
A **cross-encoder** reads the question and a chunk **together** and scores how well the chunk answers it - much more accurate, but too slow to run on every chunk.

So production RAG does two steps:
1. vector search fetches many candidates (e.g. 20)
2. the re-ranker re-orders them and keeps the best few (e.g. 3-5)

`ContextualCompressionRetriever` wraps any retriever with a post-processing step - here the re-ranker.
We use a free local cross-encoder (first run downloads it). Managed alternatives: Cohere Rerank, Jina, Voyage.

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

question = "Which food helps with eyesight?"

base_retriever = store.as_retriever(search_kwargs={"k": 6})   # step 1: many candidates
print("vector search order:")
for d in base_retriever.invoke(question):
    print("  ", d.page_content)

reranker = CrossEncoderReranker(
    model=HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2"),
    top_n=2,                                                  # step 2: keep the best 2
)
rerank_retriever = ContextualCompressionRetriever(base_compressor=reranker, base_retriever=base_retriever)

print("\nafter re-ranking:")
for d in rerank_retriever.invoke(question):
    print("  ", d.page_content)

## A3. Multi-query retriever
Users phrase questions in one way; your documents may use different words.
`MultiQueryRetriever` asks an **LLM** to rewrite the question several ways, searches with each version, and returns the **unique union** of the results.

Costs one LLM call per question. Turn on the logger to see the generated queries.

In [ ]:
import logging
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)   # prints the generated queries

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=store.as_retriever(search_kwargs={"k": 2}),
    llm=llm,
)

for d in multi_query_retriever.invoke("what should I eat for more energy?"):
    print("  ", d.page_content)

## A4. Parent document retriever
The chunking trade-off:
- **small chunks** → precise embeddings, good search, but little context for the LLM
- **big chunks** → good context, but blurry embeddings and worse search

`ParentDocumentRetriever` does both: it **searches small child chunks** but **returns the bigger parent** they came from.
- children (with embeddings) go in the **vector store**
- parents (plain text, no embeddings) go in a **docstore** (`InMemoryStore` here; Redis / a database in production)
- optional `parent_splitter`: use medium-sized sections as parents instead of whole documents

In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

parent_docs = [
    Document(
        page_content=(
            "Mangoes are tropical fruits grown mostly in India. "
            "They are rich in vitamin A and vitamin C. "
            "Alphonso is one of the most popular varieties. "
            "Mango season in India runs from April to June."
        ),
        metadata={"source": "mango_guide.txt"},
    ),
    Document(
        page_content=(
            "Spinach is a leafy green vegetable. "
            "It is high in iron, calcium and vitamin K. "
            "Cooking spinach reduces its volume a lot. "
            "It grows best in cool weather."
        ),
        metadata={"source": "spinach_guide.txt"},
    ),
]

child_store = Chroma(collection_name="vs_parent_demo", embedding_function=embeddings)   # in memory
parent_retriever = ParentDocumentRetriever(
    vectorstore=child_store,
    docstore=InMemoryStore(),
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=60, chunk_overlap=0),
)
parent_retriever.add_documents(parent_docs)

question = "When is mango season?"
print("child chunk that matched:")
print("  ", child_store.similarity_search(question, k=1)[0].page_content)

print("\nwhat the retriever returns (the whole parent):")
print("  ", parent_retriever.invoke(question)[0].page_content)

### Summary

| Technique | Extra cost | Use when |
|---|---|---|
| Indexing API | a small SQL table | data is re-ingested regularly (files, wikis, DB exports) |
| Re-ranking | cross-encoder per candidate | answer quality matters more than a little latency |
| Multi-query | 1 LLM call per question | users ask vague or differently worded questions |
| Parent document | a docstore for parents | answers need more context than one small chunk |

These combine: e.g. multi-query → parent documents → re-rank the result.